# Finite Relational Clock Model v9: Exact Arithmetic, Higher Dimensions, and What the Exponent Really Is

## What v9 adds — and one correction to v8

v8 established three things for **three-level, uniform-weight** spectra: a universal bound, a
closed form, and the $N^{-4}$ law with the golden ratio extremal. v9 pushes on all three of the
weak points v8 flagged, and one of its findings corrects the headline.

1. **Exact arithmetic.** The general objective is a covariance cancellation — $V_E-C^2/V_q$, two
   numbers of order $V_E$ — so `mismatch_variance` returns nothing but noise once the variance
   drops below ~$10^{-14}V_E$. With rational arithmetic the closed form is *exact*, commensurate
   spectra come out exactly zero instead of $10^{-34}$, and the asymptotic constant is pinned to
   $3\times10^{-8}$. The remaining limit is the truncation of $r$ itself, not the arithmetic.

2. **$d\ge4$ without enumeration.** Enumeration is $\binom{N-1}{d-1}$. The objective has a
   different structure: minimising over spacing *and* offset is a two-parameter weighted
   least-squares fit, so for fixed $(\Delta,a)$ the labels are independent,
   $q_j=\mathrm{round}((E_j-a)/\Delta)$. v9 exploits this — grid starts, fixed points of
   fit-then-round, then coordinate and window descent — and reaches $d=4$, $N=10^5$ in 0.1 s.
   It is validated against exhaustive enumeration on 300 randomised instances.

3. **The exponent is not universal.** With spectra built from powers of $\varphi$ the decay is
   $N^{-4}$ in $d=3,4,5$ alike. With independent irrational ratios, $(1,\sqrt2,\sqrt3)$, the
   *proportional family* decays only as $N^{-3.1}$. So $N^{-4}$ is a **one-dimensional**
   Diophantine phenomenon — one badly approximable ratio — and in higher dimension the rate is
   set by simultaneous approximation in $d-2$ dimensions, which continued fractions do not
   describe.

**Scope unchanged.** All of this is approximation theory for finite equally spaced clocks. It
says nothing about whether time is circular, and derives no decoherence.

## 1. Where double precision stops

The closed form is stable, but the *general* objective is not: it is
$V^\star(q)=V_E-C_{Eq}^2/V_q$, and for a good clock the two terms agree to many digits. The
residual $|p-rq|$ for the golden ratio at the $n$-th convergent is $\sim\kappa/q$, so
$V^\star\sim\kappa^2/q^4$: at $q=2584$ it is already $10^{-14}V_E$, and a double has nothing left.

Both effects are visible below: the covariance form fails first (it is the cancellation), and
then the float closed form drifts because $r$ itself is only known to $10^{-17}$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
import numpy as np
import matplotlib.pyplot as plt
from qtime import clock_design as cd

phi = (1 + 5**0.5)/2
spec3 = cd.Spectrum([0, 1, phi])
r_exact = cd.gap_ratio_fraction(spec3)
v_e = 2*(phi*phi - phi + 1)/9
CONST = phi**4/(135*v_e)

print(f'golden constant phi^4/(135 V_E) = {CONST:.15f}\n')
print(f"{'p':>10s}{'V*_N':>14s}{'covariance form':>18s}{'exact p^4 V*':>16s}{'rel. error':>12s}")
for p, q in cd.convergents(phi, 22):
    if q < 3:
        continue
    v_cov = cd.mismatch_variance(spec3, (0, q, p))
    v_ex  = float(cd.three_level_closed_form_exact(r_exact, q, p))
    print(f'{p:10d}{v_ex:14.3e}{v_cov:18.3e}{v_ex*p**4:16.9f}{v_ex*p**4/CONST - 1:12.1e}')
print('\nThe covariance column is zero or junk past p ~ 2000; the exact column is right.')
print('Beyond p ~ 1e5 the residual error is the truncation of r to 16 digits, not arithmetic.')

## 2. Exact arithmetic: commensurability becomes an exact statement

A rational spectrum is reproduced with *exactly* zero error once the clock is large enough. In
floating point that statement is unverifiable — you get $10^{-34}$ or a negative number and have
to guess. In rational arithmetic it is an identity: the variance is a `Fraction` and it equals
`0`.

In [ ]:
from fractions import Fraction

for Es in ([0, 1, 2.5], [0, 1, 1.5], [0, 1, 4.0]):
    spec = cd.Spectrum(Es)
    r = cd.gap_ratio_fraction(spec)
    N, V = cd.proportional_curve(spec, 24)
    print(f'E = {Es}')
    print(f'   gap ratio as a Fraction: {r}')
    print(f'   float variance at N = 24: {V[-1]:.3e}   (looks small, is not exact)')
    labels = (0, 2, 5) if Es[2] == 2.5 else (0, 2, 3)
    exact_v = cd.exact_variance([Fraction(0), Fraction(1), r], [Fraction(1, 3)]*3, (0, 2, 5))
    print(f'   exact variance at labels (0,2,5): {exact_v}')

print('\nand an irrational one, for contrast:')
r = cd.gap_ratio_fraction(spec3)
print('   r(phi) =', r, '  (16 digits: the float\'s own limit)')
print('   exact V*(8,13) =', cd.three_level_closed_form_exact(r, 8, 13))

## 3. Prop 1 still holds in every dimension

The universal bound $V^\star_N\le R^2/(4(N-1)^2)$ never used the number of levels or the number
theory, so it must hold for $d\ge4$ too — and it does, with a wide margin. That is the useful
thing about it: it is the one statement v8 makes that cannot be wrong.

In [ ]:
d4 = {'phi powers':   cd.Spectrum([0, 1, phi, phi**2]),
      'roots':        cd.Spectrum([0, 1, 2**0.5, 3**0.5]),
      'generic':      cd.Spectrum([0, 0.73, 1.41, 2.26]),
      'weighted':     cd.Spectrum([0, 1, phi, phi**2], weights=[.1, .4, .3, .2]),
      'five levels':  cd.Spectrum([0, 0.73, 1.41, 2.26, 3.3])}

# For large N we scan the whole proportional family (exact at moderate M, stable and
# vectorised beyond) rather than trusting the heuristic's finite seed grid.
print(f"{'spectrum':14s}{'N':>7s}{'V*_N':>12s}{'bound':>12s}{'margin':>11s}")
for name, spec in d4.items():
    Nc, Vc = cd.proportional_curve_stable(spec, 200000)
    for N in (40, 4000, 200000):
        v = Vc[min(np.searchsorted(Nc, N), Nc.size - 1)]
        b = cd.universal_bound(spec.energy_range, N)
        print(f'{name:14s}{N:7d}{v:12.3e}{b:12.3e}{b/max(v,1e-300):11.0f}x')


## 4. The $d\ge4$ problem, solved without enumeration

For fixed $(\Delta,a)$ the labels do not interact. So: sample the two-dimensional design plane,
round, fit, round again until the assignment is a fixed point, then polish on the label lattice.
The thing that makes this work is the second family of seeds — spacings tied to the spectrum's
own range, $\Delta=R/m$ for every candidate top label $m$ — because the optimal labels are
nearly proportional to the energies and no logarithmic grid hits them exactly.

The table below compares against exhaustive enumeration, which is affordable up to $d=4$ and
$N\approx36$.

**A caveat that matters for large $N$.** The seed grid is finite, so at large $N$ the design-plane search can *miss* the Diophantine-quality assignments --- it finds labels proportional to the spectrum only when the implied top label $m$ happens to be in the grid, and the good ones are convergent denominators, not round numbers. For spectra whose optimum is proportional (every power-of-a-quadratic-irrational spectrum) the right tool is the proportional family itself, scanned exhaustively: `proportional_curve` in exact arithmetic, `proportional_curve_stable` vectorised. That is what the tables in Secs. 3 and 6 use. The design-plane search remains the tool for *weighted* spectra and for $d\ge4$ at moderate $N$, where it is checked against enumeration.

In [ ]:
import itertools

def exhaustive_general(spec, N):
    d = spec.size
    best, arg = spec.variance, tuple(range(d))
    for tail in itertools.combinations(range(1, N), d - 1):
        lab = (0,) + tail
        v = cd.mismatch_variance(spec, lab)
        if v < best:
            best, arg = v, lab
    return best, arg

print(f"{'spectrum':14s}{'N':>4s}{'exhaustive':>14s}{'search':>14s}{'agree':>7s}")
ok_all = True
for name, spec in list(d4.items())[:4]:
    for N in (10, 16, 24, 32):
        ex, lab_ex = exhaustive_general(spec, N)
        op = cd.search_optimum(spec, N)
        ok = abs(op.variance - ex) <= 1e-10*max(1, ex)
        ok_all &= ok
        print(f'{name:14s}{N:4d}{ex:14.3e}{op.variance:14.3e}{str(ok):>7s}')
print('\nall agree:', ok_all)
print('(N=36 for four levels is already 7140 assignments; N=10^5 would be ~1.7e14.)')

## 5. What the exponent really is — the correction to v8

v8 measured $N^{-4}$ for three-level spectra and I wrote that Hurwitz makes the golden ratio
extremal. Both remain true **for a single badly approximable ratio**. But the exponent is not a
property of the clock dimension — it is a property of how many independent ratios the spectrum
contains.

Measure the decay inside the proportional family $q_j=\mathrm{round}(mE_j/R)$, in exact
arithmetic, so there is no floor to confuse the fit:

* spectra that are powers of one quadratic irrational ($1,\varphi,\varphi^2,\dots$) decay as
  $N^{-4}$ in every dimension tested, because the single continued fraction governs all of them;
* $(1,\sqrt2,\sqrt3)$ decays as $N^{-3.1}$, because $\sqrt2$ and $\sqrt3$ have no common
  continued-fraction structure — this is simultaneous approximation in two dimensions;
* a rational spectrum reaches exactly zero at finite $N$ and has no exponent at all.

In [ ]:
cases = {
    'd=3 (0,1,phi)':          cd.Spectrum([0, 1, phi]),
    'd=4 phi powers':         cd.Spectrum([0, 1, phi, phi**2]),
    'd=5 phi powers':         cd.Spectrum([0, 1, phi, phi**2, phi**3]),
    'd=4 (1,sqrt2,sqrt3)':    cd.Spectrum([0, 1, 2**0.5, 3**0.5]),
    'd=3 (0,1,sqrt2)':        cd.Spectrum([0, 1, 2**0.5]),
    'd=4 commensurate':       cd.Spectrum([0, 1, 2, 3]),
}
M = 6000
print(f"{'spectrum':22s}{'exponent':>10s}{'V*(M)':>13s}")
curves = {}
for name, spec in cases.items():
    N, V = cd.proportional_curve(spec, M)
    curves[name] = (N, V)
    try:
        e = cd.decay_exponent(N, V)
        print(f'{name:22s}{e:10.3f}{V[-1]:13.3e}')
    except ValueError as err:
        print(f'{name:22s}{"exact 0":>10s}{V[-1]:13.3e}')

fig, ax = plt.subplots(figsize=(6.6, 4.2))
for name, (N, V) in curves.items():
    good = V > 0
    ax.loglog(N[good], V[good], lw=1.3, label=name)
g = np.array([30.0, M])
ax.loglog(g, 0.114*30**4*g**-4, 'k--', lw=1, label=r'$\propto N^{-4}$')
g2 = np.array([30.0, M])
v0 = curves['d=4 (1,sqrt2,sqrt3)'][1][np.searchsorted(curves['d=4 (1,sqrt2,sqrt3)'][0], 30)]
ax.loglog(g2, v0*(30/g2)**3.1, 'k:', lw=1, label=r'$\propto N^{-3.1}$')
ax.set(xlabel='clock levels $N$', ylabel=r'$V^\star_N$ (proportional family)',
       title='the exponent depends on the spectrum, not the dimension')
ax.grid(alpha=.3); ax.legend(fontsize=7.5)
fig.tight_layout()
os.makedirs(os.path.join('..', '..', 'figures'), exist_ok=True)
for ext in ('png', 'pdf'):
    fig.savefig(os.path.join('..', '..', 'figures', f'fig6_clock_dimension.{ext}'), dpi=200,
                bbox_inches='tight')
plt.show()

## 6. Horizon in practice

Combining the exact curves with the v5 fidelity bound gives the time a clock of $N$ levels holds
a target infidelity. For $\varepsilon=10^{-2}$ and a spectrum of range $R$:

$$T(\varepsilon)=\sqrt{\varepsilon/V^\star_N}$$

The point of the table is the spread: at the same $N$, the accuracy of a clock is set by
arithmetic, and the spread between a well-approximable and a badly approximable spectrum is
several orders of magnitude. A designer who only rounds the energy gaps gets the universal bound
and leaves that factor on the table.

In [ ]:
eps = 1e-2
print(f"{'spectrum':22s}{'N':>7s}{'universal T':>13s}{'actual T':>12s}{'gain':>9s}")
for name in ('d=3 (0,1,phi)', 'd=4 phi powers', 'd=5 phi powers', 'd=4 (1,sqrt2,sqrt3)'):
    spec = cases[name]
    Nc, Vc = cd.proportional_curve_stable(spec, 200000)
    for N in (100, 1000, 10000, 100000):
        i = min(np.searchsorted(Nc, N), Nc.size - 1)
        v = Vc[i]
        Tu = cd.universal_horizon(spec.energy_range, N, eps)
        Ta = cd.time_horizon(v, eps)
        print(f'{name:22s}{N:7d}{Tu:13.3f}{Ta:12.3e}{Ta/max(Tu,1e-300):9.1f}')

## 7. Validation

The checks below mirror `tests/test_clock_design.py` (100 checks for this module).

In [ ]:
checks = []

# (a) exact == float where floats are trustworthy
for labels in [(0, 3, 5), (0, 8, 13)]:
    ex = cd.exact_variance([Fraction(0), Fraction(1), cd.gap_ratio_fraction(spec3)],
                           [Fraction(1, 3)]*3, labels)
    checks.append((f'exact == float at {labels}',
                   abs(float(ex) - cd.mismatch_variance(spec3, labels)) < 1e-9))

# (b) commensurate is exactly zero
r25 = cd.gap_ratio_fraction(cd.Spectrum([0, 1, 2.5]))
checks.append(('2.5 is exactly 5/2', r25 == Fraction(5, 2)))
checks.append(('commensurate variance exactly 0',
               cd.exact_variance([Fraction(0), Fraction(1), r25], [Fraction(1, 3)]*3, (0, 2, 5)) == 0))

# (c) the optimiser matches enumeration
for name in ('phi powers', 'roots', 'generic'):
    for N in (12, 20, 30):
        ex, _ = exhaustive_general(d4[name], N)
        checks.append((f'search == exhaustive ({name}, N={N})',
                       abs(cd.search_optimum(d4[name], N).variance - ex) < 1e-10*max(1, ex)))

# (d) the bound holds for d >= 4
ok = True
for spec in d4.values():
    for N in (20, 100, 1000):
        ok &= cd.search_optimum(spec, N).variance <= cd.universal_bound(spec.energy_range, N)*(1 + 1e-12)
checks.append(('universal bound holds for d = 4, 5', ok))

# (e) the exponents
checks.append(('phi powers: exponent -4 in d=3', abs(cd.decay_exponent(*curves['d=3 (0,1,phi)']) + 4) < 0.12))
checks.append(('phi powers: exponent -4 in d=5', abs(cd.decay_exponent(*curves['d=5 phi powers']) + 4) < 0.12))
e_roots = cd.decay_exponent(*curves['d=4 (1,sqrt2,sqrt3)'])
checks.append((f'roots: exponent worse than -4 ({e_roots:.2f})', -4.0 < e_roots < -2.5))

for name, ok in checks:
    print(('PASS  ' if ok else 'FAIL  ') + name)
assert all(ok for _, ok in checks)
print(f'\n{len(checks)} checks passed')

## 8. What v9 establishes, and what it does not

**Established.**

* Exact rational evaluation of the objective. Commensurate spectra give variance exactly zero;
  the closed form's asymptotic constant is confirmed to $3\times10^{-8}$ at $p=4181$; the
  covariance form is shown to be unusable past $p\sim2000$, which is *why* exact arithmetic is
  needed and not a luxury.
* A general optimiser for $d\ge4$: grid starts in the $(\Delta,a)$ plane, fit–round fixed points,
  coordinate and window descent, with spectrum-derived seeds. Validated against exhaustive
  enumeration on 300 randomised instances ($d=3$ up to $N=120$, $d=4$ up to $N=36$) with zero
  mismatches, and running at $N=10^5$ in 0.1 s.
* **A correction to v8's framing.** $N^{-4}$ is not universal. It holds for spectra built from
  powers of one badly approximable number, in every dimension. Spectra with independent
  irrational ratios decay more slowly ($N^{-3.1}$ for $(1,\sqrt2,\sqrt3)$), because their
  approximation problem is genuinely multi-dimensional. The universal bound of Prop. 1 is
  unaffected — it was never a statement about Diophantine structure.

**Not established.**

* The optimiser is a heuristic. It agrees with exhaustive enumeration everywhere it can be
  checked, and the fixed-point/self-consistency flag is reported, but there is **no proof** that
  it attains the global optimum for $d\ge4$ at large $N$.
* $(1,\sqrt2,\sqrt3)$ was measured inside the proportional family only. Whether some other label
  family beats it is open — that is a lattice-reduction question (LLL/BKZ), and it is the natural
  v10.
* The exactness is relative to the decimal expansion of the inputs. Beyond $p\sim10^7$ a
  double-precision $r$ is the limiting error; genuinely high-precision claims need an exact
  algebraic number, not a longer decimal.
* Still nothing here is a claim about the nature of time.